In [ ]:
import wntr
import numpy as np

import networkx as nx

from collections import OrderedDict

import math

from numpy.linalg import inv
import scipy.integrate as spi
import matplotlib.pyplot as plt


# plt.style.use("visualization/FST.mplstyle")

In [ ]:
def construct_L_matrix(
    wn, edge_dict, g=9.81, default_link_length=1, default_link_diameter=0.2
):
    c = len(edge_dict)  #  cardinality
    L = np.zeros((c, c))
    for i, link_name in enumerate(edge_dict.values()):
        link = wn.get_link(link_name)
        if link_name in wn.pipe_name_list:
            L[i, i] = 4 * link.length / (g * np.pi * link.diameter**2)
        else:
            L[i, i] = 4 * default_link_length / (g * np.pi * default_link_diameter**2)
    return L


def construct_C_vector(
    wn, edge_dict, g=9.81, a=1000, default_link_length=1, default_link_diameter=0.2
):
    c = len(edge_dict)  # cardinality
    C = np.zeros(c)
    for i, link_name in enumerate(edge_dict.values()):
        link = wn.get_link(link_name)
        if link_name in wn.pipe_name_list:
            C[i] = (g * np.pi / 4 * link.diameter**2 * link.length) / a**2
        else:
            C[i] = (
                g * np.pi / 4 * default_link_diameter**2 * default_link_length
            ) / a**2
    return C


def construct_R_matrix(
    wn,
    edge_dict,
    g=9.81,
    loss_model="hw",
    default_link_length=1,
    default_link_diameter=0.2,
    default_link_roughness=100,
):
    c = len(edge_dict)  #  cardinality
    R = np.zeros((c, c))
    if loss_model == "hw":
        for i, link_name in enumerate(edge_dict.values()):
            link = wn.get_link(link_name)
            if link_name in wn.pipe_name_list:
                R[i, i] = (
                    10.666829500036352
                    * link.length
                    / (link.roughness**1.852 * link.diameter**4.871)
                )
            else:
                R[i, i] = (
                    10.666829500036352
                    * default_link_length
                    / (default_link_roughness**1.852 * default_link_diameter**4.871)
                )
    elif loss_model == "dw":
        for i, link_name in enumerate(edge_dict.values()):
            link = wn.get_link(link_name)
            if link_name in wn.pipe_name_list:
                R[i, i] = (
                    8 * link.roughness * link.length / (np.pi**2 * g * link.diameter**5)
                )
            else:
                R[i, i] = (
                    8
                    * default_link_roughness
                    * default_link_length
                    / (np.pi**2 * g * default_link_diameter**5)
                )
    else:
        raise ValueError("loss model not allowed")
    return R


def construct_CT_I_vector(wn, internal_node_list):
    CT = np.zeros(len(internal_node_list))
    for i, node_name in enumerate(internal_node_list):
        node = wn.get_node(node_name)
        if node.node_type == "Tank":
            CT[i] = math.pi * node.diameter**2 / 4
        elif node.node_type == "Reservoir":
            CT[i] = math.pi * 100**2 / 4
    return CT


def construct_Q_vector(wn, internal_node_list):
    Q = np.zeros(len(internal_node_list))
    for i, node_name in enumerate(internal_node_list):
        node = wn.get_node(node_name)
        if node.node_type == "Junction":
            Q[i] = node.base_demand
    return Q


In [ ]:
wn = wntr.network.WaterNetworkModel()

wn.add_reservoir("N1", base_head=1.4)
wn.add_junction("N2", elevation=0.875, base_demand=0.0)
wn.add_junction("N3", elevation=0.875, base_demand=0.0)
wn.add_junction("N4", elevation=0.83, base_demand=0.0001)
# wn.add_junction("N5", elevation=85, base_demand=240 / 3600)


wn.add_pipe(
    "P1",
    start_node_name="N1",
    end_node_name="N2",
    length=1.7,
    diameter=0.015,
    roughness=0.000007,
)
wn.add_pipe(
    "P2",
    start_node_name="N2",
    end_node_name="N3",
    length=0.1,
    diameter=0.015,
    roughness=0.000007,
)
wn.add_pipe(
    "P3",
    start_node_name="N3",
    end_node_name="N4",
    length=0.35,
    diameter=0.015,
    roughness=0.000007,
)
# wn.add_pipe(
#     "P4",
#     start_node_name="N4",
#     end_node_name="N5",
#     length=1000,
#     diameter=0.3,
#     roughness=130,
# )

mdG = wn.to_graph()
G = nx.DiGraph(mdG)

wn.options.hydraulic.demand_model = "DD"
wn.options.hydraulic.headloss = "D-W"
wn.options.hydraulic.minimum_pressure = 0.001
wn.options.hydraulic.required_pressure = 0.001
wn.options.time.duration = 1000
wn.options.time.hydraulic_timestep = 1
wn.options.time.report_timestep = 1
sim = wntr.sim.EpanetSimulator(wn)
results = sim.run_sim()

In [ ]:
results.node["head"].iloc[0]

In [ ]:
mdG = wn.to_graph(
    link_weight=results.link["flowrate"].iloc[0], modify_direction=True
)  # returns a multidigraph

G = nx.DiGraph(mdG)

internal_node_list = list(wn.junction_name_list) + list(wn.tank_name_list)
reservoir_list = list(wn.reservoir_name_list)

edge_list = list(G.edges())
edge_dict = OrderedDict(
    {
        edge: [
            link_name
            for link_name, link in wn.links()
            if (link.start_node_name == edge[0] and link.end_node_name == edge[1])
            or (link.start_node_name == edge[1] and link.end_node_name == edge[0])
        ][0]
        for edge in edge_list
    }
)

x = OrderedDict(
    {"q": list(edge_dict.values()), "h_I": internal_node_list, "h_R": reservoir_list}
)  # vector of state variables: x = [q; h], see van Gemert

x_dim = len(x["q"]) + len(x["h_I"]) + len(x["h_R"])
# B, D = np.zeros(x_dim).reshape((x_dim, 1)), 0

# incidence matrix in the paper has opposite signs than the one nx returns by default
A_inc = -nx.incidence_matrix(
    G, nodelist=internal_node_list + reservoir_list, edgelist=edge_list, oriented=True
)

A_inc_I = A_inc[: len(internal_node_list)]
A_inc_R = A_inc[-len(reservoir_list) :]

L = construct_L_matrix(wn, edge_dict, default_link_length=4, default_link_diameter=0.2)
L_inv = inv(L)
Cv = construct_C_vector(
    wn, edge_dict, a=500, default_link_length=4, default_link_diameter=0.2
)
R = construct_R_matrix(
    wn,
    edge_dict,
    loss_model="dw",
    default_link_roughness=0.00002,
    default_link_diameter=0.2,
    default_link_length=4,
)
CT_I = construct_CT_I_vector(wn, internal_node_list)
Q_I = construct_Q_vector(wn, internal_node_list)

fv = 1 / (0.98 * np.pi * 0.1**2 / 4 * 1) ** 2 / (2 * 9.81)
Fv = np.diag([fv if i == "P4" else 0 for i in x["q"]])

discharge = 0  # 0.00136372
D_I = np.array([discharge if i == "1" else 0 for i in x["h_I"]])

M = inv(np.diag(1 / 2 * np.abs(A_inc_I) @ Cv + CT_I))


In [ ]:
M

In [ ]:
# set initial conditions

h_R = np.array([wn.get_node(r).base_head for r in x["h_R"]]).reshape(1, len(x["h_R"]))


y_eq_q = []
for k in x["q"]:
    edge = wn.get_link(k)
    if (edge.start_node.name, edge.end_node.name) in edge_dict.keys():
        y_eq_q.append(results.link["flowrate"].iloc[0][k])
    elif (edge.end_node.name, edge.start_node.name) in edge_dict.keys():
        y_eq_q.append(-results.link["flowrate"].iloc[0][k])
    else:
        print("didnt work for {}".format(k))

y_eq_h = [results.node["head"].iloc[0][k] for k in x["h_I"]]
y_eq = np.concatenate([y_eq_q, y_eq_h])

q_0 = y_eq[: len(x["q"])]  # y0_steady[: len(x["q"])]
h_0 = y_eq[-len(x["h_I"]) :]  # y0_steady[-len(x["h_I"]) :]

Q_1 = -L_inv @ R @ np.diag(np.abs(q_0))
Q_2 = L_inv @ A_inc_I.T
Q_3 = -M @ A_inc_I
Q_4 = -M @ D_I * np.diag(1 / (2 * np.sqrt(h_0)))


A = np.block([[Q_1, Q_2], [Q_3, Q_4]])
B_q = L_inv @ A_inc_R.T.todense() @ h_R[0]

In [ ]:
A

In [ ]:
def ewcm_state(
    t,
    y,
    delta_h_tol=-1e-6,  # to activate, change to positive
    delta_q_tol=-1e-10,  # to activate, change to positive
):
    q = y[: len(x["q"])]
    h = y[-len(x["h_I"]) :]
    # V = valve_closure(t, Fv, t_valve_start, t_valve_end)

    Q_t = Q_I
    B = np.block([B_q, -M @ Q_t])
    dydt = A @ y + B

    return dydt


In [ ]:
def integrate(
    t, f, x_0, dt, args={}, tol=1e-15
):  # Runge-Kutta 45 https://de.wikipedia.org/wiki/Klassisches_Runge-Kutta-Verfahren
    k1 = f(t, x_0, **args)
    k2 = f(t + dt / 2, x_0 + dt / 2 * k1, **args)
    k3 = f(t + dt / 2, x_0 + dt / 2 * k2, **args)
    k4 = f(t + dt, x_0 + dt * k3, **args)
    return x_0 + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


In [ ]:
y_eq_r = np.array(
    [
        0.0001000000,
        0.0001000000,
        0.00010000000,
        1.372537523110129,
        1.3709220837892178,
        1.3652680432885076,
    ]
)  # i got this but averaging the results from the next cell

In [ ]:
dt = 1e-4
t_vec = np.arange(0, 20, dt)

x_true = y_eq_r  # or just use y_eq (wntr result)
x_hist = []

for t in t_vec:
    x_true = integrate(t, ewcm_state, x_true, dt)
    x_hist.append(x_true)

In [ ]:
x_hist = np.array(x_hist)

fig, axs = plt.subplots(len(x_hist[0]), figsize=(5, 20))

for i in range(len(x_hist[0])):
    axs[i].plot(x_hist[:, i])

In [ ]:
sol = spi.solve_ivp(
    ewcm_state,
    [0, 10],
    y_eq,  # y0_zeng,
    method="Radau",
    # t_eval=np.linspace(0, t_end, n_steps),
    # first_step=1e-8,
    jac=A,
    atol=1e-8,  # [1e-8 if i > 0.1 else 1e-14 for i in y_eq],
    rtol=1e-8,  # [1e-8 if i > 0.1 else 1e-16 for i in y_eq],
    max_step=1e-3,
)

In [ ]:
y_eq

In [ ]:
fig, axs = plt.subplots(len(sol.y.T[0]), figsize=(5, 20))

for i in range(len(sol.y.T[0])):
    axs[i].plot(sol.y.T[:, i])